In [1]:
pip install kornia segmentation-models-pytorch

  Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 3.4 MB/s eta 0:00:01
   ------------------- -------------------- 0.5/1.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------- ----------- 0.8/1.1 MB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 1.3 MB/s  0:00:00
Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl (154 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ------------ --------------------------- 0.8/2.5 MB 3.7 MB/s eta 0:00:01
   ------------ --------------------------- 0.8/2.5 MB 3.7 MB/s eta 0:00:01
   ---------------- ----------------------- 1.0

In [3]:
pip install rasterio

  Using cached rasterio-1.4.3-cp311-cp311-win_amd64.whl.metadata (9.4 kB)
  Using cached affine-2.4.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached rasterio-1.4.3-cp311-cp311-win_amd64.whl (25.5 MB)
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
Using cached affine-2.4.0-py3-none-any.whl (15 kB)
Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)

   ------------- -------------------------- 2/6 [affine]
   --------------------------------- ------ 5/6 [rasterio]
   --------------------------------- ------ 5/6 [rasterio]
   --------------------------------- ------ 5/6 [rasterio]
   --------------------------------- ------ 5/6 [rasterio]
   ---------------------------------------- 6/6 [rasterio]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import rasterio
from pathlib import Path
import kornia.augmentation as K
import kornia.augmentation.container as C
from torch.amp import autocast, GradScaler

# SMP Imports
from segmentation_models_pytorch.decoders.upernet.decoder import UPerNetDecoder
from segmentation_models_pytorch.base import SegmentationHead

# --- MODIFIED: Import Summit/MAE dependencies instead of DOFA ---
import mae_model
from util.pos_embed import interpolate_pos_embed

# ============================================================================
# 1. Feature Extraction Helper (MODIFIED for MAE)
# ============================================================================

class ViTFeatureExtractor:
    def __init__(self, model, hook_indices=[3, 5, 7, 11]):
        self.model = model
        self.hook_indices = hook_indices
        self.features = {}
        self.hooks = []
        
        # Summit MAE also uses 'blocks', so this logic remains valid
        for idx in hook_indices:
            layer = model.blocks[idx]
            self.hooks.append(layer.register_forward_hook(self._get_hook(f'block_{idx}')))

    def _get_hook(self, name):
        def hook(model, input, output):
            self.features[name] = output
        return hook

    def clear(self):
        self.features = {}

    def remove_hooks(self):
        for h in self.hooks:
            h.remove()

def extract_patch_features(model, images, feature_extractor):
    feature_extractor.clear()
    
    # --- MODIFIED SECTION START ---
    # DOFA used: _ = model(images, wavelengths)
    # Summit MAE uses: forward_encoder with mask_ratio=0
    
    # We use mask_ratio=0 to ensure the encoder sees the WHOLE image (no masking)
    # The hooks inside FeatureExtractor will capture the intermediate features
    _ = model.forward_encoder(images, mask_ratio=0)
    # --- MODIFIED SECTION END ---
    
    return feature_extractor.features

c:\Users\91983\miniconda3\envs\summit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================================
# 2. Decoder (KEPT EXACTLY AS DOFA CODE)
# ============================================================================

class SummitUPerNet(nn.Module):
    def __init__(self, encoder_dim=768, decoder_channels=256, num_classes=2, patch_size=16, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        encoder_channels = [3, encoder_dim, encoder_dim, encoder_dim, encoder_dim]
        
        self.feature_proj = nn.ModuleDict({
            'block_3': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'block_5': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'block_7': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'block_11': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
        })
        
        self.decoder = UPerNetDecoder(
            encoder_channels=encoder_channels,
            encoder_depth=4,
            decoder_channels=decoder_channels,
            use_norm="batchnorm",
        )
        
        self.segmentation_head = SegmentationHead(
            in_channels=decoder_channels,
            out_channels=num_classes,
            activation=None,
            kernel_size=1,
            upsampling=4, 
        )
        self.dropout = nn.Dropout2d(dropout)
    
    def reshape_vit_features(self, features, H, W):
        B, N, D = features.shape
        # MAE also has a CLS token at index 0, so we skip it
        if N == (H * W) + 1: features = features[:, 1:, :]
        return features.transpose(1, 2).reshape(B, D, H, W)
    
    def forward(self, features_dict, target_size, dummy_input=None):
        B, N, D = features_dict['block_11'].shape
        H = W = int(np.sqrt(N)) if N % int(np.sqrt(N)) == 0 else int(np.sqrt(N - 1))
        
        feat_3 = self.feature_proj['block_3'](self.reshape_vit_features(features_dict['block_3'], H, W))
        feat_5 = self.feature_proj['block_5'](self.reshape_vit_features(features_dict['block_5'], H, W))
        feat_7 = self.feature_proj['block_7'](self.reshape_vit_features(features_dict['block_7'], H, W))
        feat_11 = self.feature_proj['block_11'](self.reshape_vit_features(features_dict['block_11'], H, W))
        
        if dummy_input is None: dummy_input = torch.zeros(B, 3, H, H, device=feat_3.device)
        else: dummy_input = F.interpolate(dummy_input, size=(H, H), mode='bilinear', align_corners=False)
        
        features_list = [dummy_input, feat_3, feat_5, feat_7, feat_11]
        x = self.segmentation_head(self.dropout(self.decoder(features_list)))
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)


In [ ]:
# ============================================================================
# 3. Trainer & Utils (SLIGHT MODS FOR INPUT CHANNELS)
# ============================================================================

# Adjusted stats for 3-channel SAR (VV, VH, Avg)
S1_MEAN = [166.36, 88.45, 127.41] 
S1_STD = [64.83, 43.07, 53.95]

class GPUAugmentation(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        # 1. Geometric Augmentations (Applied to BOTH Image and Mask)
        self.aug = C.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            data_keys=["input", "mask"], # synchronized transform
            same_on_batch=False
        )
        
        # 2. Normalization (Applied ONLY to Image manually)
        self.normalize = K.Normalize(mean=torch.tensor(mean), std=torch.tensor(std))

    def forward(self, img, mask):
        # Apply geometry to both (flips happen in sync)
        img, mask = self.aug(img, mask)
        
        # Apply normalization ONLY to the image
        img = self.normalize(img)
        
        return img, mask

class SegmentationTrainer:
    def __init__(self, vit_model, decoder, train_loader, device='cuda', 
                 encoder_lr=1e-5, decoder_lr=3e-4, num_epochs=100, checkpoint_dir='checkpoints/Summit+UPerNet'):
        self.vit_model = vit_model.to(device)
        self.decoder = decoder.to(device)
        self.train_loader = train_loader
        # Removed self.wavelengths (Not needed for MAE)
        self.device = device
        self.num_epochs = num_epochs
        self.checkpoint_dir = checkpoint_dir
        
        self.extractor_hook = ViTFeatureExtractor(self.vit_model, hook_indices=[3, 5, 7, 11])
        self.augmentor = GPUAugmentation(mean=S1_MEAN, std=S1_STD).to(device)
        
        self.vit_model.train()
        for param in self.vit_model.parameters(): param.requires_grad = True 
        
        self.optimizer = AdamW([
            {'params': self.vit_model.parameters(), 'lr': encoder_lr},
            {'params': self.decoder.parameters(), 'lr': decoder_lr}
        ], weight_decay=0.01)
        
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=num_epochs)
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = GradScaler()
        self.best_train_iou = 0.0
        self.history = []
        os.makedirs(checkpoint_dir, exist_ok=True)
    
    def train_epoch(self, epoch):
        self.decoder.train()
        self.vit_model.train()
        total_loss, total_iou = 0.0, 0.0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.num_epochs}')
        for batch in pbar:
            images = batch['image'].to(self.device, non_blocking=True)
            masks = batch['mask'].to(self.device, non_blocking=True).float()
            if len(masks.shape) == 3:
                masks = masks.unsqueeze(1)
            
            images, masks = self.augmentor(images,masks)
            masks = masks.long().squeeze(1)
            images = images.contiguous()
            masks = masks.contiguous()

            with autocast('cuda'):
                # --- MODIFIED: Removed wavelengths argument ---
                features_dict = extract_patch_features(self.vit_model, images, self.extractor_hook)
                logits = self.decoder(features_dict, masks.shape[-2:], dummy_input=images)
                loss = self.criterion(logits, masks)
            
            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            loss_item = loss.item()
            iou = self.compute_iou(logits.detach(), masks)
            total_loss += loss_item
            total_iou += iou
            
            pbar.set_postfix({'loss': f'{loss_item:.4f}', 'IoU': f'{iou:.4f}'})
        
        return {'loss': total_loss / len(self.train_loader), 'iou': total_iou / len(self.train_loader)}

    def compute_iou(self, logits, masks):
        preds = torch.argmax(logits, dim=1)
        num_classes = logits.shape[1]
        iou_per_class = []
        for cls in range(num_classes):
            pred_cls = (preds == cls)
            mask_cls = (masks == cls)
            intersection = (pred_cls & mask_cls).sum().float()
            union = (pred_cls | mask_cls).sum().float()
            if union > 0: iou_per_class.append((intersection / union).item())
        return np.mean(iou_per_class) if iou_per_class else 0.0
    
    def train(self):
        for epoch in range(self.num_epochs):
            metrics = self.train_epoch(epoch)
            self.scheduler.step()
            print(f"\nEpoch {epoch+1} - Loss: {metrics['loss']:.4f}, IoU: {metrics['iou']:.4f}")
            self.history.append({'epoch': epoch+1, **metrics})
            
            if metrics['iou'] > self.best_train_iou:
                self.best_train_iou = metrics['iou']
                self.save_checkpoint(epoch, metrics, is_best=True)
            
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch, metrics, is_best=False)

    def save_checkpoint(self, epoch, metrics, is_best=False):
        checkpoint = {
            'epoch': epoch,
            'encoder_state_dict': self.vit_model.state_dict(),
            'decoder_state_dict': self.decoder.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metrics': metrics,
        }
        torch.save(checkpoint, os.path.join(self.checkpoint_dir, f'ckpt_epoch_{epoch}.pth'))
        if is_best:
            torch.save(checkpoint, os.path.join(self.checkpoint_dir, 'best_model.pth'))

In [ ]:
# ============================================================================
# 4. Dataset (MINIMAL CHANGE: 2 Ch -> 3 Ch FIX)
# ============================================================================

class SegmentationDataset(Dataset):
    def __init__(self, root_dir: str):
        self.root_dir = Path(root_dir)
        self.mask_paths = sorted(list(self.root_dir.glob("labels/*.png")))
        self.vv_dir = self.root_dir / "vv"
        self.vh_dir = self.root_dir / "vh"

    def __len__(self):
        return len(self.mask_paths)

    def __getitem__(self, idx):
        mask_path = self.mask_paths[idx]
        filename = mask_path.name
        
        with rasterio.open(self.vv_dir / filename) as f:
            vv = f.read().astype('float32')
        with rasterio.open(self.vh_dir / filename) as f:
            vh = f.read().astype('float32')
        with rasterio.open(mask_path) as f:
            label = f.read().astype('float32')
        
        label = np.where(label > 128, 1, 0)

        # --- MODIFIED: Ensure 3 Channels for MAE ---
        # Summit Weights expect 3 channels. You have 2. 
        # We append the average as the 3rd channel.
        avg_ch = (vv + vh) / 2.0
        s1_img = np.concatenate((vv, vh, avg_ch), axis=0)
        
        return {"image": torch.from_numpy(s1_img), "mask": torch.from_numpy(label).long()}

In [ ]:
def main():
    config = {
        "encoder_dim": 768, "decoder_channels": 256, "num_classes": 2,
        "patch_size": 16, "image_size": 224, 
        "batch_size": 100,  # Reduced batch size for safety
        "encoder_lr": 1e-5, "decoder_lr": 3e-4, "num_epochs": 100,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "head_dropout": 0.1
    }
    
    # --- MODIFIED: Load Summit Checkpoint ---
    print("Loading Summit ViT Encoder...")
    
    # 1. Path to your Summit Checkpoint
    checkpoint_path = r"C:\Users\91983\Downloads\summit_checkpoint_ViT-b.pth"
    
    # 2. Instantiate MAE Model at 224x224
    vit_model = mae_model.mae_vit_base_patch16(img_size=config['image_size'])
    
    # 3. Load & Interpolate
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    if 'model' in checkpoint:
        state_dict = checkpoint['model']
    else:
        state_dict = checkpoint

    print("Interpolating Pos Embeddings (448 -> 224)...")
    interpolate_pos_embed(vit_model, state_dict)
    
    msg = vit_model.load_state_dict(state_dict, strict=False)
    print("Checkpoint Loaded:", msg)

    print("Preparing Data...")
    # Update this path to your dataset
    train_dataset = SegmentationDataset(root_dir=r'C:\Users\91983\Documents\ARM\tiled_dataset')
    
    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=4
    )

    print("Initializing Decoder...")
    decoder = SummitUPerNet(
        encoder_dim=config['encoder_dim'],
        decoder_channels=config['decoder_channels'],
        num_classes=config['num_classes'],
        dropout=config['head_dropout'],
        patch_size=config['patch_size']
    )

    print(f"Starting Training (Batch Size: {config['batch_size']})...")
    trainer = SegmentationTrainer(
        vit_model=vit_model,
        decoder=decoder,
        train_loader=train_loader,
        device=config['device'],
        encoder_lr=config['encoder_lr'],
        decoder_lr=config['decoder_lr'],
        num_epochs=config['num_epochs']
    )
    trainer.train()

if __name__ == "__main__":
    main()